In [ ]:
import os
import json
import random
import itertools
import math

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
import lightning as L

import matplotlib.pyplot as plt
from matplotlib import colormaps
from IPython.display import display, HTML
from tabulate import tabulate

import warnings
import gc

from models import get_model
from data import get_dataset
from utils.plot import *
from xai import *
from metrics import *
# plt.rcParams["font.size"] = 24
# plt.rcParams["font.family"] = "cmr10"

os.environ["TOKENIZERS_PARALLELISM"] = "true"


In [ ]:
device = "cuda:0"
model_dir = "/home/xia/claim-xai/configs/flickr8k/base"
# model_dir = "/home/xia/claim-xai/configs/hateful-memes/base"
# model_dir = "/home/xia/claim-xai/configs/snli-ve/base"
model_config = json.load( open(os.path.join(model_dir, "config.json"), "r") )
model_metadata = json.load( open(os.path.join(model_dir, "masker/_metadata.json"), "r") )

In [ ]:
ds_train, ds_val, ds_test, num_classes = get_dataset(
    model_config["dataset_name"], 
    data_dir = "../_datasets", 
    splits = ["train", "validation", "test"],
    **model_config["dataset_args"], 
)


model = get_model(
    model_name = model_config["model_name"],
    checkpoint_path = os.path.join(model_dir, model_metadata["best_checkpoint"]),
    **model_config["model_args"]
).eval().to(device)

---

In [ ]:
for i in range(5):
    image, text, label = ds_test[i]

    with torch.no_grad():
        image_inputs, text_inputs = model.preprocess([image], [text])
        logits, text_weights, image_weights = model(image_inputs, text_inputs)

    tokens = text_inputs["input_ids"].cpu().squeeze(0)
    text_weights = text_weights.cpu()

    image_weights = image_weights[0].cpu()


    print(f"pred={torch.argmax(logits).item()} label={label}")

    attr_plot = text_weights.mean(dim=2)[0]
    attr_plot = (attr_plot - attr_plot.min()) / (attr_plot.max() - attr_plot.min() + 1e-16)
    attr_plot = attr_plot.tolist()
    display_text_attribution(tokens[1:], attr_plot[1:], model.text_encoder._model.tokenizer, "<pad>")

    image_plot = image.cpu()
    attr_plot = torchvision.transforms.functional.resize(image_weights.cpu().detach(), (image_plot.shape[1], image_plot.shape[2]), interpolation=torchvision.transforms.InterpolationMode.BILINEAR)
    attr_plot = (attr_plot - attr_plot.min()) / (attr_plot.max() - attr_plot.min())
    plot_image_attribution(image_plot, attr_plot)
    plt.show()

---

In [ ]:
image, text, label = ds_test[51]

with torch.no_grad():
    image_inputs, text_inputs = model.preprocess([image], [text])
    logits, text_weights, image_weights = model(image_inputs, text_inputs)
pred = torch.argmax(logits).item()
print(f"pred={pred} label={label}")

tokens = text_inputs["input_ids"].cpu().squeeze(0)
tokens = tokens.tolist()


inputs = (image_inputs["pixel_values"], text_inputs["input_ids"], text_inputs["attention_mask"])
text_embeds = model.text_encoder.embed(inputs[1])

baselines_text = []
baselines_image = []
for i in range(20):
    image_inputs, text_inputs = model.preprocess([ds_train[i][0]], [ds_train[i][1]])
    text_embeds = model.text_encoder.embed(text_inputs["input_ids"])
    baselines_text.append(text_embeds)
    baselines_image.append(image_inputs["pixel_values"])
baselines_text = torch.cat(baselines_text)
baselines_image = torch.cat(baselines_image)

explainer_our = OurAttribution(model)
attr_our = explainer_our(inputs)

explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0]*0, inputs[1]*0))
attr_lig = explainer_lig((inputs[0], inputs[1]), target=pred, additional_forward_args=inputs[2])

explainer_saliency = SaliencyAttribution(model)
attr_saliency = explainer_saliency((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_dl = DeepLiftAttribution(model, baselines=(inputs[0]*0.0, text_embeds*0.0))
    attr_dl = explainer_dl((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_dlshap = DeepLiftShapAttribution(model, baselines=(baselines_image, baselines_text))
    attr_dlshap = explainer_dlshap((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2].unsqueeze(1))

explainer_grad_shap = GradientShapAttribution(model, baselines=(baselines_image, baselines_text))
attr_grad_shap = explainer_grad_shap((inputs[0], text_embeds), target=pred, n_samples=100, additional_forward_args=inputs[2])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_gbprop = GuidedBackpropAttribution(model)
    attr_gbprop = explainer_gbprop((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2])

for attr in [ attr_our, attr_lig, attr_saliency, attr_dl, attr_dlshap, attr_grad_shap, attr_gbprop ]:
    attr_text_plot = attr[1].mean(dim=2)[0]
    attr_text_plot = (attr_text_plot - attr_text_plot.min()) / (attr_text_plot.max() - attr_text_plot.min())
    attr_text_plot = attr_text_plot.tolist()
    display_text_attribution(tokens[1:], attr_text_plot[1:], model.text_encoder._model.tokenizer, "<pad>")

    attr_image_plot = attr[0].mean(dim=1, keepdim=True)
    attr_image_plot = torchvision.transforms.functional.resize(attr_image_plot[0].cpu().detach(), (image_plot.shape[1], image_plot.shape[2]), interpolation=torchvision.transforms.InterpolationMode.BILINEAR)
    attr_image_plot = (attr_image_plot - attr_image_plot.min()) / (attr_image_plot.max() - attr_image_plot.min())
    image_plot = image.cpu()
    plot_image_attribution(image_plot, attr_image_plot)
    plt.show()

---

In [ ]:
def _accum_metrics(metrics, model_base, inputs, attribution):
    metrics["avg-drop"].accumulate(model_base, inputs, attribution)
    metrics["delete-auc"].accumulate(model_base, inputs, attribution)
    metrics["complexity"].accumulate(attribution)
    metrics["sparsity"].accumulate(attribution)

In [ ]:
L.seed_everything(42)
model_base = get_model_wrapper(model)

methods = [ "ours", "layer-ig", "saliency", "deeplift", "deeplift-shap", "gradient-shap", "guided-backprop" ]
metrics = {
    method: {
        "avg-drop": AverageDropMetric(device),
        "delete-auc": DeletionAUCMetric(device),
        "complexity": ComplexityMetric(device),
        "sparsity": SparsityMetric(device)
    } for method in methods
}


for i in tqdm(range(len(ds_test))):
    image, text, label = ds_test[i]

    with torch.no_grad():
        image_inputs, text_inputs = model.preprocess([image], [text])
        logits, text_weights, image_weights = model(image_inputs, text_inputs)
    pred = torch.argmax(logits).item()

    inputs = (image_inputs["pixel_values"], text_inputs["input_ids"], text_inputs["attention_mask"])
    text_embeds = model.text_encoder.embed(inputs[1])

    baselines_text = []
    baselines_image = []
    for i in range(20):
        image_inputs, text_inputs = model.preprocess([ds_train[i][0]], [ds_train[i][1]])
        text_embeds = model.text_encoder.embed(text_inputs["input_ids"])
        baselines_text.append(text_embeds)
        baselines_image.append(image_inputs["pixel_values"])
    baselines_text = torch.cat(baselines_text)
    baselines_image = torch.cat(baselines_image)

    explainer_our = OurAttribution(model)
    attr_our = explainer_our(inputs)
    _accum_metrics(metrics["ours"], model_base, inputs, attr_our)

    explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0]*0, inputs[1]*0))
    attr_lig = explainer_lig((inputs[0], inputs[1]), target=pred, additional_forward_args=inputs[2])
    _accum_metrics(metrics["layer-ig"], model_base, inputs, attr_lig)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_saliency = SaliencyAttribution(model)
        attr_saliency = explainer_saliency((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2])
        _accum_metrics(metrics["saliency"], model_base, inputs, attr_saliency)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_dl = DeepLiftAttribution(model, baselines=(inputs[0]*0.0, text_embeds*0.0))
        attr_dl = explainer_dl((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2])
        _accum_metrics(metrics["deeplift"], model_base, inputs, attr_dl)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_dlshap = DeepLiftShapAttribution(model, baselines=(baselines_image, baselines_text))
        attr_dlshap = explainer_dlshap((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2].unsqueeze(1))
        _accum_metrics(metrics["deeplift-shap"], model_base, inputs, attr_dlshap)

    explainer_grad_shap = GradientShapAttribution(model, baselines=(baselines_image, baselines_text))
    attr_grad_shap = explainer_grad_shap((inputs[0], text_embeds), target=pred, n_samples=100, additional_forward_args=inputs[2])
    _accum_metrics(metrics["gradient-shap"], model_base, inputs, attr_grad_shap)
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_gbprop = GuidedBackpropAttribution(model)
        attr_gbprop = explainer_gbprop((inputs[0], text_embeds), target=pred, additional_forward_args=inputs[2])
        _accum_metrics(metrics["guided-backprop"], model_base, inputs, attr_gbprop)

    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
tab_out = []
for method in metrics:
    out = [ method ]
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        out.append( f"{mean:.2f} +/- {std:.2f}" )
    tab_out.append( out )

print(tabulate(
    tab_out, 
    headers = ["method", *metrics["ours"]]
))

In [ ]:
json_out = {}
for method in metrics:
    json_out[method] = {}
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        json_out[method][metric] = { "mean": mean, "std": std }

In [ ]:
json.dump(json_out, open(os.path.join(model_dir, "metrics.json"), "w"), indent=4)